**Theme:**

*“A neural network is just stacked linear layers + non-linearity.”*

# Imports

In [1]:
import torch 
import torch.nn as nn 

torch.__version__

'2.8.0+cu129'

In [2]:
import torchvision 
from torchvision import datasets, transforms 
from torch.utils.data import DataLoader 

# My first *ANN* **(MNIST)**

## Define the Artificial Neural Network

In [8]:
transform = transforms.ToTensor()

# Load MNIST  
train_data = datasets.MNIST(
    root='data', train=True, download=True, transform=transform 
)

test_data = datasets.MNIST(
    root='data', train=False, download=True, transform=transform
)

train_loader = DataLoader(train_data, batch_size=100, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=100, shuffle=False) 

# Neural Network 
class MLP(nn.Module):
    def __init__(self):
        super().__init__()

        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128), 
            nn.ReLU(), 
            nn.Linear(128, 10)
        )
    def forward(self,x):
        return self.net(x) 
    
model =MLP() 
loss_fn = nn.CrossEntropyLoss() 
optimizer = torch.optim.Adam(model.parameters(), lr=0.001) 

## One training batch

In [9]:
images, labels = next(iter(train_loader)) 

# forward pass 
outputs = model(images)  
# compute loss
loss = loss_fn(outputs, labels)  

# zero grad
optimizer.zero_grad() 
# backward pass 
loss.backward() 
# parameters update 
optimizer.step() 

print("Loss: ", loss.item())

Loss:  2.297685146331787


- What is the shape of outputs?

- Why does CrossEntropyLoss need 10 outputs?

In [10]:
outputs.shape

torch.Size([100, 10])

## Full training loop

In [11]:
for i in range(5): 
    total_loss = 0.0 

    for images, labels in train_loader:

        outputs = model(images) 
        loss = loss_fn(outputs, labels)  
        total_loss += loss.item()

        optimizer.zero_grad()  
        loss.backward() 
        optimizer.step() 

    print(f"Epoch: {i+1}, Loss: {total_loss/len(train_loader) :.6f}")

Epoch: 1, Loss: 0.386211
Epoch: 2, Loss: 0.179114
Epoch: 3, Loss: 0.126472
Epoch: 4, Loss: 0.097537
Epoch: 5, Loss: 0.077846


- Why do we average the loss?

- What does decreasing loss mean here?

## Evaluate the accuracy of the trained model

In [8]:
correct = 0 
total = 0 

with torch.no_grad():
    for images, labels in test_loader: 
        outputs = model(images) 
        
        preds = outputs.argmax(dim=1) 

        correct += (preds==labels).sum().item() 
        total += labels.size(0) 

print(f"Test Accuracy: {correct/total :.5f}")

Test Accuracy: 0.97390


- Why use `argmax`?

- What does this accuracy represent?

# Interaction

### What does `argmax(dim=1)` really mean?


outputs has shape:
```bash
(batch_size, num_classes) = (64, 10)
```
Example for one image:
```bash
[ 2.1, -0.5, 0.3, 5.2, 1.1, ... ]

Each number = model’s confidence for a digit.
```
argmax finds:

- the index of the largest value

So:
```bash
[2.1, -0.5, 0.3, 5.2, ...] → 3
```
Meaning: **digit 3**

#### Why dim=1?

Because:
```bash
dim=0 → across batches (wrong)
dim=1 → across classes (correct)
```
Let’s visualize:
```bash
outputs =
[
  [scores for image 1]  ← dim=1
  [scores for image 2]
  ...
]
```
We want:

- best class per image

So:
```python
preds = outputs.argmax(dim=1)
```
returns:
```bash
[3, 7, 1, 0, 4, ...]   # one digit per image
```


### What if you do NOT give `dim`?


```python
outputs.argmax()
```

This:

- Flattens the entire tensor

- Finds the biggest number among all 640 values

- Returns one single index

This is useless for classification.

- It does NOT give per-image predictions.

> Interview-ready answer

argmax(dim=1) selects the most likely class for each sample by finding the highest logit across the class dimension.